# Section 1: Load CCSR Features + Build Target Variable


In [1]:
import pandas as pd
import numpy as np
import duckdb
import warnings
warnings.filterwarnings('ignore')

# Load CCSR features built by teammate
# Shape: 9996 patients x 706 columns (subject_id + 705 CCSR flags)
ccsr = pd.read_parquet('outputs/ccsr_features.parquet')
print(f"CCSR features loaded: {ccsr.shape}")
print(f"First few columns: {list(ccsr.columns[:5])}")
print(ccsr.head())

CCSR features loaded: (9996, 706)
First few columns: ['subject_id', "'1    '", "'10   '", "'100  '", "'101  '"]
   subject_id  '1    '  '10   '  '100  '  '101  '  '102  '  '103  '  '104  '  \
0    10997834        0        0        0        0        0        0        0   
1    12378205        0        0        0        0        0        0        0   
2    17826349        0        0        0        0        0        0        0   
3    14142053        0        0        0        0        0        0        0   
4    12950664        0        0        0        0        0        0        0   

   '105  '  '106  '  ...  'SYM010'  'SYM011'  'SYM012'  'SYM013'  'SYM014'  \
0        0        0  ...         0         0         0         0         0   
1        0        0  ...         0         0         0         0         0   
2        0        0  ...         0         0         1         0         0   
3        0        0  ...         0         0         0         0         0   
4        0       

# Section 2: Build Target Variable and Join with CCSR Features


In [2]:

con = duckdb.connect()

# Build readmitted_30d target from admissions
target_query = """
WITH all_admissions AS (
    SELECT 
        a.subject_id,
        a.hadm_id,
        a.admittime,
        a.dischtime,
        a.admission_type,
        a.insurance,
        a.race,
        a.discharge_location,
        a.marital_status,
        p.gender,
        p.anchor_age + (
            EXTRACT(year FROM CAST(a.admittime AS TIMESTAMP)) 
            - p.anchor_year
        ) AS age_at_admission,
        DATE_DIFF('day',
            CAST(a.admittime AS TIMESTAMP),
            CAST(a.dischtime AS TIMESTAMP)
        ) AS los_days,
        CASE WHEN a.edregtime IS NOT NULL THEN 1 ELSE 0 END AS came_via_ed,
        ROW_NUMBER() OVER (
            PARTITION BY a.subject_id ORDER BY a.admittime
        ) - 1 AS prior_admissions_count,
        LEAD(a.admittime) OVER (
            PARTITION BY a.subject_id ORDER BY a.admittime
        ) AS next_admittime
    FROM 'data/admissions.csv' a
    JOIN 'data/patients.csv' p ON a.subject_id = p.subject_id
    WHERE a.subject_id IN (
        SELECT CAST(subject_id AS BIGINT) 
        FROM 'outputs/cohort_10k.csv'
    )
    AND a.hospital_expire_flag = 0
),
with_target AS (
    SELECT *,
        DATE_DIFF('day',
            CAST(dischtime AS TIMESTAMP),
            CAST(next_admittime AS TIMESTAMP)
        ) AS days_to_next,
        CASE
            WHEN next_admittime IS NULL THEN 0
            WHEN DATE_DIFF('day',
                CAST(dischtime AS TIMESTAMP),
                CAST(next_admittime AS TIMESTAMP)
            ) <= 30 THEN 1
            ELSE 0
        END AS readmitted_30d
    FROM all_admissions
)
SELECT
    subject_id,
    hadm_id,
    gender,
    age_at_admission,
    admission_type,
    insurance,
    race,
    discharge_location,
    marital_status,
    los_days,
    came_via_ed,
    prior_admissions_count,
    days_to_next,
    readmitted_30d
FROM with_target
"""

df_target = con.execute(target_query).df()

print(f"Target table shape: {df_target.shape}")
print(f"\nTarget distribution:")
print(df_target['readmitted_30d'].value_counts())
print(f"\nReadmission rate: {df_target['readmitted_30d'].mean()*100:.1f}%")
print(df_target.head())

Target table shape: (23896, 14)

Target distribution:
0    19177
1     4719
Name: readmitted_30d, dtype: int64

Readmission rate: 19.7%
   subject_id   hadm_id gender  age_at_admission               admission_type  \
0    10014765  29840268      M                71               EU OBSERVATION   
1    10014765  26650343      M                77            OBSERVATION ADMIT   
2    10022862  21636831      M                50  SURGICAL SAME DAY ADMISSION   
3    10022862  27262738      M                51  SURGICAL SAME DAY ADMISSION   
4    10027407  21216166      M                63                     EW EMER.   

  insurance   race discharge_location marital_status  los_days  came_via_ed  \
0   Private  WHITE               None        MARRIED         1            1   
1  Medicare  WHITE               HOME        MARRIED         5            1   
2   Private  WHITE               HOME        MARRIED         6            0   
3   Private  WHITE               HOME        MARRIED         

# Section 3: Join Target + CCSR Features into One Modeling Table


In [3]:

# The CCSR table has one row per patient (subject_id level)
# The target table has one row per admission (hadm_id level)
# We merge on subject_id — each admission gets the patient's CCSR flags

df_model = df_target.merge(ccsr, on='subject_id', how='left')

print(f"Modeling table shape: {df_model.shape}")
print(f"Columns: {len(df_model.columns)} total")
print(f"\nTarget distribution:")
print(df_model['readmitted_30d'].value_counts())
print(f"\nReadmission rate: {df_model['readmitted_30d'].mean()*100:.1f}%")
print(f"\nMissing values (top 10):")
print(df_model.isnull().sum().sort_values(ascending=False).head(10))
print(f"\nFirst 3 rows:")
print(df_model.head(3))

Modeling table shape: (23896, 719)
Columns: 719 total

Target distribution:
0    19177
1     4719
Name: readmitted_30d, dtype: int64

Readmission rate: 19.7%

Missing values (top 10):
days_to_next          9783
discharge_location    6577
marital_status         510
insurance              403
subject_id               0
'INJ045'                 0
'INJ035'                 0
'INJ036'                 0
'INJ037'                 0
'INJ038'                 0
dtype: int64

First 3 rows:
   subject_id   hadm_id gender  age_at_admission               admission_type  \
0    10014765  29840268      M                71               EU OBSERVATION   
1    10014765  26650343      M                77            OBSERVATION ADMIT   
2    10022862  21636831      M                50  SURGICAL SAME DAY ADMISSION   

  insurance   race discharge_location marital_status  los_days  ...  'SYM010'  \
0   Private  WHITE               None        MARRIED         1  ...         0   
1  Medicare  WHITE             

In [7]:
df_model.head(10)
#df_model.describe()

,subject_id,hadm_id,gender,age_at_admission,admission_type,insurance,race,discharge_location,marital_status,los_days,...,'SYM010','SYM011','SYM012','SYM013','SYM014','SYM015','SYM016','SYM017','SYM018','XXX000'
0,10014765,29840268,M,71,EU OBSERVATION,Private,WHITE,None,MARRIED,1,...,0,0,0,0,0,0,0,1,0,1
1,10014765,26650343,M,77,OBSERVATION ADMIT,Medicare,WHITE,HOME,MARRIED,5,...,0,0,0,0,0,0,0,1,0,1
2,10022862,21636831,M,50,SURGICAL SAME DAY ADMISSION,Private,WHITE,HOME,MARRIED,6,...,0,0,0,0,0,0,0,0,0,1
3,10022862,27262738,M,51,SURGICAL SAME DAY ADMISSION,Private,WHITE,HOME,MARRIED,1,...,0,0,0,0,0,0,0,0,0,1
4,10027407,21216166,M,63,EW EMER.,Private,WHITE,HOME,MARRIED,1,...,0,0,0,0,0,0,0,0,0,0
5,10028500,28989837,M,60,EU OBSERVATION,Private,WHITE,None,MARRIED,1,...,0,0,0,0,0,0,0,0,0,0
6,10046436,23594537,M,31,URGENT,Private,WHITE,HOME,SINGLE,5,...,0,0,0,0,0,0,0,0,0,1
7,10046436,21447783,M,34,OBSERVATION ADMIT,Medicaid,WHITE,OTHER FACILITY,SINGLE,20,...,0,0,0,0,0,0,0,0,0,1
8,10056441,28971129,F,45,EW EMER.,Other,HISPANIC OR LATINO,HOME,SINGLE,5,...,0,0,0,0,0,0,0,0,0,0
9,10078747,22567903,F,20,EU OBSERVATION,Private,BLACK/AFRICAN AMERICAN,None,SINGLE,0,...,0,0,0,0,0,0,0,0,0,0
